In [1]:
import warnings

warnings.filterwarnings("ignore")


In [2]:
import pandas as pd

df1 = pd.read_csv("../../data/cleaned_dataset.csv", sep="|", engine="python")       # les textes
df2_init = pd.read_csv("../../data/paires_mot_lemme.csv", )                         # les mots

df2 = df2_init.rename(columns={'mot': 'old_french', 'lemme': 'modern'})



In [3]:
df1["type"] = "phrase"
df1.head()
df1.shape


(6082, 3)

In [4]:
df2["type"] = "mot"
df2.head()
df2.shape


(32878, 3)

In [5]:
df = pd.concat([df1, df2], ignore_index=True)
df.shape

(38960, 3)

In [6]:
# Création des colonnes 
df["input_text"] = df["modern"].astype(str)
df["target_text"] = df["old_french"].astype(str)



In [7]:
# Supprime les NaN classiques
df = df.dropna()

# Supprime les lignes où "nan" apparaît comme string (dans input_text ou target_text)
mask = (df["input_text"].str.lower() != "traduire: nan") & (df["target_text"].str.lower() != "nan")
df = df[mask].reset_index(drop=True)


In [8]:
# On garde juste le nécessaire
df = df[["input_text", "target_text", "type"]]
print(df.sample(3))

                                             input_text  \
4482  cher ami(e), je voulais vous confier que, réce...   
3256  yo, c'est kévin du 93, je suis obsédé par les ...   
1292  "oh la la, les émotions, c'est bien compliqué,...   

                                            target_text    type  
4482  chier ami(e), je vos voloie fere saveir que, n...  phrase  
3256  io, c'est kévin del 93, jo suif obsédé par les...  phrase  
1292  hélas, les émotions, moult sont compliquées, s...  phrase  


In [9]:
# Adapter pour MarianMT
df = df.rename(columns={"input_text": "src", "target_text": "tgt"})


In [10]:
from datasets import Dataset, Features, ClassLabel, Value

features = Features({
    "src": Value("string"),
    "tgt": Value("string"),
    "type": ClassLabel(names=["phrase", "mot"])
})

dataset = Dataset.from_pandas(df, features=features)

dataset = dataset.train_test_split(test_size=0.1, seed=42, stratify_by_column="type")



In [11]:
# supprimer la colonne "type" du dataset final pour ne pas l'utiliser pour l'entraînement :

dataset["train"] = dataset["train"].remove_columns("type")
dataset["test"] = dataset["test"].remove_columns("type")

In [12]:
from transformers import MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-fr-en"  # Base légère, fine-tuning sur ta tâche

tokenizer = MarianTokenizer.from_pretrained(model_name)
max_length = 400

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["src"],
        max_length=max_length,
        padding="max_length",
        truncation=True
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["tgt"],
            max_length=max_length,
            padding="max_length",
            truncation=True
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)


Map: 100%|██████████| 3523/3523 [00:00<00:00, 4823.60 examples/s]


In [13]:
from transformers import MarianMTModel, DataCollatorForSeq2Seq

model = MarianMTModel.from_pretrained(model_name)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)


2025-06-22 15:17:27.754464: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-22 15:17:27.760597: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750598247.769223   16259 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750598247.771768   16259 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750598247.778210   16259 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [14]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./marianmt-vieux-francais2",
    per_device_train_batch_size=10,
    per_device_eval_batch_size=10,
    num_train_epochs=10,
    eval_strategy="steps",
    save_steps=500,
    logging_steps=100,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)
trainer.train()


Step,Training Loss,Validation Loss
100,0.759200,0.238650
200,0.252400,0.210558
300,0.232200,0.191343
400,0.196400,0.183208
500,0.190700,0.175260
600,0.190100,0.168058
700,0.186900,0.163705
800,0.168500,0.159159
900,0.158900,0.154883
1000,0.174900,0.151422


TrainOutput(global_step=31710, training_loss=0.08269775460022777, metrics={'train_runtime': 17074.866, 'train_samples_per_second': 18.566, 'train_steps_per_second': 1.857, 'total_flos': 3.3581627080704e+16, 'train_loss': 0.08269775460022777, 'epoch': 10.0})

In [15]:
# sauvegarde du modele

trainer.save_model("./marianmt-vieux-francais-model2")
tokenizer.save_pretrained("./marianmt-vieux-francais-model2")


('./marianmt-vieux-francais-model2/tokenizer_config.json',
 './marianmt-vieux-francais-model2/special_tokens_map.json',
 './marianmt-vieux-francais-model2/vocab.json',
 './marianmt-vieux-francais-model2/source.spm',
 './marianmt-vieux-francais-model2/target.spm',
 './marianmt-vieux-francais-model2/added_tokens.json')

In [1]:
import torch

def generate_translation_marian(text, model, tokenizer, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    batch = tokenizer.prepare_seq2seq_batch([text], return_tensors="pt", max_length=128, truncation=True)
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        gen = model.generate(**batch, max_length=128, num_beams=4)
    return tokenizer.decode(gen[0], skip_special_tokens=True)

test_text = "Bonjour à vous, je suis ravi."
result = generate_translation_marian(test_text, model, tokenizer)
print("Moderne :", test_text)
print("Vieux français :", result)


NameError: name 'model' is not defined

In [3]:
test_text = "en cette journée ensoléillée ou il fait très chaud, je vais aller me baigner à la mer car j'ai trop chaud."
result = generate_translation_marian(test_text, model, tokenizer)
print("Moderne :", test_text)
print("Vieux français :", result)


/home/malek/BRIEFS DEV IA/14.NLP/EULA-vaaag-/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:4085: FutureWarning: 
`prepare_seq2seq_batch` is deprecated and will be removed in version 5 of HuggingFace Transformers. Use the regular
`__call__` method to prepare your inputs and targets.

Here is a short example:

model_inputs = tokenizer(src_texts, text_target=tgt_texts, ...)

If you either need to use different keyword arguments for the source and target texts, you should do two calls like
this:

model_inputs = tokenizer(src_texts, ...)
labels = tokenizer(text_target=tgt_texts, ...)
model_inputs["labels"] = labels["input_ids"]

See the documentation of your specific tokenizer for more details on the specific arguments to the tokenizer of choice.
For a more complete example, see the implementation of `prepare_seq2seq_batch`.

  warnings.warn(formatted_warning, FutureWarning)


Moderne : en cette journée ensoléillée ou il fait très chaud, je vais aller me baigner à la mer car j'ai trop chaud.
Vieux français : en ceste journée ensoleillée ou il fait moult chault, je vais aller me bainner à la mer car j'ai trop chault.


In [7]:
from transformers import MarianMTModel, MarianTokenizer

# Chargement du modèle et du tokenizer fine-tunés
model_path = "./marianmt-vieux-francais-model2"
model = MarianMTModel.from_pretrained(model_path)
tokenizer = MarianTokenizer.from_pretrained(model_path)


In [10]:
test_text = "WTF ! Encore une fois, j'ai failli me casser la gueule en me précipitant pour trouver les chiottes sur le bus.  J'avais trop mal au ventre, j'étais sûre que j'allais cracher mon petit déjeuner, j'étais tellement stressée sur le trajet."
test_text2 = "En plus, la musique était trop nulle, on dirait que le chauffeur avait une collection de Céline Dion sur le vieux téléphone portable de son grand-père. Enfin, j'ai pu m'évacuer, mais j'ai failli rater mon arrêt, la vie est un enfer !"

result = generate_translation_marian(test_text, model, tokenizer)
result2 = generate_translation_marian(test_text2, model, tokenizer)

print("Moderne :", test_text)
print("Vieux français :", result)

print("Moderne :", test_text2)
print("Vieux français :", result2)



/home/malek/BRIEFS DEV IA/14.NLP/EULA-vaaag-/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:4085: FutureWarning: 
`prepare_seq2seq_batch` is deprecated and will be removed in version 5 of HuggingFace Transformers. Use the regular
`__call__` method to prepare your inputs and targets.

Here is a short example:

model_inputs = tokenizer(src_texts, text_target=tgt_texts, ...)

If you either need to use different keyword arguments for the source and target texts, you should do two calls like
this:

model_inputs = tokenizer(src_texts, ...)
labels = tokenizer(text_target=tgt_texts, ...)
model_inputs["labels"] = labels["input_ids"]

See the documentation of your specific tokenizer for more details on the specific arguments to the tokenizer of choice.
For a more complete example, see the implementation of `prepare_seq2seq_batch`.

  warnings.warn(formatted_warning, FutureWarning)


Moderne : WTF ! Encore une fois, j'ai failli me casser la gueule en me précipitant pour trouver les chiottes sur le bus.  J'avais trop mal au ventre, j'étais sûre que j'allais cracher mon petit déjeuner, j'étais tellement stressée sur le trajet.
Vieux français : "vivre ! encore une foiz, faillie moi brisier en me hastant por trover les latrines sur le char. aveie trop dolour al ventre, esteie certe que alerai escracher mon disner, esteie tant esmaie sur le chemin."
Moderne : En plus, la musique était trop nulle, on dirait que le chauffeur avait une collection de Céline Dion sur le vieux téléphone portable de son grand-père. Enfin, j'ai pu m'évacuer, mais j'ai failli rater mon arrêt, la vie est un enfer !
Vieux français : Plus, la musique estoit trop mauvais, on dirait que le charretier avoit une collection de Céline Dion sur le vieil téléphonie portable de son grand-père. finalement, j'ay peu me vacuer, mais j'ay failli faillir mon arrêt, la vie est un enfer !


In [ ]:
# Moderne : WTF ! Encore une fois, j'ai failli me casser la gueule en me précipitant pour trouver les chiottes sur le bus.  J'avais trop mal au ventre, j'étais sûre que j'allais cracher mon petit déjeuner, j'étais tellement stressée sur le trajet.
# Vieux français : "vivre ! encore une foiz, faillie moi brisier en me hastant por trover les latrines sur le char. aveie trop dolour al ventre, esteie certe que alerai escracher mon disner, esteie tant esmaie sur le chemin."

# Moderne : En plus, la musique était trop nulle, on dirait que le chauffeur avait une collection de Céline Dion sur le vieux téléphone portable de son grand-père. Enfin, j'ai pu m'évacuer, mais j'ai failli rater mon arrêt, la vie est un enfer !
# Vieux français : Plus, la musique estoit trop mauvais, on dirait que le charretier avoit une collection de Céline Dion sur le vieil téléphonie portable de son grand-père. finalement, j'ay peu me vacuer, mais j'ay failli faillir mon arrêt, la vie est un enfer !


# fine tuning